# Platform e2e
Provisions a single-node cluster, runs the landing notebook on it, refreshes the pipeline, runs the gold job and deletes the cluster. The same scenario as `dbx-ops e2e` in CI.

In [ ]:
%pip install -q databricks-sdk==0.152.0

In [ ]:
dbutils.library.restartPython()

In [ ]:
import os
import sys
from dataclasses import asdict

from databricks.sdk import WorkspaceClient

sys.path.insert(0, os.path.abspath("../src"))
from dbx_ops import ops
from dbx_ops.bundle import OrdersBundle
from dbx_ops.cli import e2e

w = WorkspaceClient()
me = w.current_user.me().user_name

dbutils.widgets.text("bundle_files", f"/Workspace/Users/{me}/.bundle/lab8_orders_cicd/prod/files")
dbutils.widgets.text("catalog", "dbr_dev_ua_5816_trail_3")
dbutils.widgets.text("bronze_schema", "lena066636_orders_bronze")
dbutils.widgets.text("landing_volume", "landing")
dbutils.widgets.text("pipeline_name", "[lena066636] orders-bronze-silver")
dbutils.widgets.text("gold_job_name", "[lena066636] orders-gold-dq")
dbutils.widgets.text("node_type", "")
dbutils.widgets.dropdown("keep_cluster", "no", ["no", "yes"])

In [ ]:
bundle = OrdersBundle(
    file_path=dbutils.widgets.get("bundle_files"),
    catalog=dbutils.widgets.get("catalog"),
    bronze_schema=dbutils.widgets.get("bronze_schema"),
    landing_volume=dbutils.widgets.get("landing_volume"),
    pipeline_id=ops.find_pipeline_id(w, dbutils.widgets.get("pipeline_name")),
    jobs={"orders_gold_dq": ops.find_job_id(w, dbutils.widgets.get("gold_job_name"))},
)
bundle

In [ ]:
report = e2e(w, bundle, dbutils.widgets.get("node_type") or None, dbutils.widgets.get("keep_cluster") == "yes")
display(spark.createDataFrame([asdict(s) for s in report.steps]))

In [ ]:
if not report.ok:
    raise RuntimeError(report.to_markdown())